In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2023_Aya_Nagar_Delhi_IMD_2023.xlsx")

In [3]:
df.head()

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,134.0,180.0,136.0,59.0,75.0,66.0,36.0,NaN,110.0,108.0,NaN,341.0
1,2,300.0,106.0,196.0,65.0,55.0,NaN,51.0,NaN,NaN,110.0,NaN,344.0
2,3,354.0,113.0,116.0,139.0,NaN,96.0,99.0,NaN,104.0,111.0,NaN,249.0
3,4,285.0,138.0,156.0,82.0,61.0,131.0,110.0,87.0,NaN,151.0,380.0,225.0
4,5,263.0,NaN,78.0,94.0,NaN,118.0,78.0,171.0,NaN,169.0,NaN,264.0


In [4]:
df.drop_duplicates
df.isnull().sum()

Day           1
January       8
February     11
March         5
April         9
May          16
June         15
July          9
August       17
September    18
October      23
November     16
December     11
dtype: int64

In [5]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape


(41, 13)

In [6]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))


In [7]:
# Define a function for outlier handling
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            # Replace outliers with mean
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())


In [8]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready


,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,134.000000,180.0,136.0,59.00000,75.00,66.000000,36.00000,94.125,110.000000,149.833333,249.96,341.0
1,2,300.000000,106.0,196.0,65.00000,55.00,82.653846,51.00000,94.125,88.565217,149.833333,249.96,344.0
2,3,354.000000,113.0,116.0,139.00000,114.84,96.000000,99.00000,94.125,104.000000,149.833333,249.96,249.0
3,4,285.000000,138.0,156.0,82.00000,61.00,82.653846,110.00000,87.000,88.565217,149.833333,380.00,225.0
4,5,263.000000,138.1,78.0,94.00000,114.84,118.000000,78.00000,94.125,88.565217,149.833333,249.96,264.0
5,6,357.000000,138.1,98.0,86.00000,114.84,93.000000,58.00000,94.125,80.000000,149.833333,249.96,205.0
6,7,314.000000,138.1,137.0,107.00000,114.84,82.653846,71.28125,82.000,93.000000,149.833333,249.96,236.7
7,8,328.000000,107.0,174.0,133.00000,112.00,82.653846,67.00000,89.000,90.000000,149.833333,249.96,236.7
8,9,411.000000,135.0,87.0,142.00000,150.00,115.000000,52.00000,103.000,91.000000,149.833333,249.96,236.7
9,10,377.000000,133.0,123.0,108.53125,162.00,102.000000,41.00000,94.125,88.565217,149.833333,198.00,235.0
